# Sionna RT — Getting Started

This notebook demonstrates the core capabilities of NVIDIA Sionna RT running on DGX Spark GB10.

**What you can do:**
- Load 3D urban scenes (Munich, Etoile, Street Canyon)
- Place transmitters and receivers
- Compute ray-traced radio propagation paths
- Generate coverage maps
- Analyze channel characteristics (delay spread, path loss, coherence bandwidth)

In [ ]:
import os
os.environ['DRJIT_LIBLLVM_PATH'] = '/usr/lib/aarch64-linux-gnu/libLLVM-14.so'

import sionna.rt as rt
import mitsuba as mi
import numpy as np

print(f'Mitsuba variant: {mi.variant()}')
print(f'Available variants: {mi.variants()}')

## Load a Scene

Sionna RT includes several built-in urban scenes. Munich is the most detailed.

In [ ]:
scene = rt.load_scene(rt.scene.munich)
print(f'Scene objects: {len(scene.objects)}')
print(f'Object names: {list(scene.objects.keys())[:10]}...')

## Configure Antennas and Devices

In [ ]:
# Set up isotropic antenna arrays
ant = rt.PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='iso', polarization='V'
)
scene.tx_array = ant
scene.rx_array = ant

# Place a transmitter (e.g., gNB on a rooftop)
tx = rt.Transmitter(name='tx0', position=[8.5, 21.0, 27.0])
scene.add(tx)

# Place a receiver (e.g., UE at street level)
rx = rt.Receiver(name='rx0', position=[45.0, 90.0, 1.5])
scene.add(rx)

print(f'TX: {tx.position.numpy()}')
print(f'RX: {rx.position.numpy()}')

## Ray Tracing — Compute Propagation Paths

The `PathSolver` traces rays through the scene, computing all paths from TX to RX
including reflections, diffractions, and scattering up to `max_depth` bounces.

In [ ]:
import time

solver = rt.PathSolver()
t0 = time.time()
paths = solver(scene, max_depth=5)
elapsed = time.time() - t0

vertices = np.array(paths.vertices)
types = np.array(paths.types)
tau = np.array(paths.tau)

print(f'Computed in {elapsed:.3f}s')
print(f'Vertices shape: {vertices.shape}')
print(f'Path types: {np.unique(types)}')
print(f'  0=LoS, 1=reflected, 2=diffracted, 3=scattered')
print(f'Delay range: {tau.min()*1e9:.1f} — {tau.max()*1e9:.1f} ns')

## Coverage Map — Radio Map Computation

The `RadioMapSolver` computes a 2D coverage map showing path gain across the scene.
This is the core tool for coverage planning and interference analysis.

In [ ]:
import matplotlib.pyplot as plt

rms = rt.RadioMapSolver()
t0 = time.time()
rm = rms(scene, cell_size=[5.0, 5.0], max_depth=3)
elapsed = time.time() - t0

print(f'Coverage map computed in {elapsed:.3f}s')

# Extract and plot
data = np.array(rm.path_gain).squeeze()
data_db = np.where(data > 0, 10 * np.log10(data + 1e-30), -200)
data_db = np.clip(data_db, -160, 0)

plt.figure(figsize=(10, 8))
plt.imshow(data_db, cmap='jet', interpolation='bilinear', origin='lower')
plt.colorbar(label='Path Gain (dB)')
plt.title('Munich — Radio Coverage Map')
plt.xlabel('X cells (5m)')
plt.ylabel('Y cells (5m)')
plt.tight_layout()
plt.show()

## Channel Analysis

From the computed paths, derive key channel metrics:
- **RMS delay spread** — determines if OFDM is needed
- **Coherence bandwidth** — max flat-fading bandwidth
- **Power delay profile** — power distribution across delays

In [ ]:
# Recompute paths for analysis
paths = solver(scene, max_depth=5)
tau = np.array(paths.tau).flatten()
tau = tau[tau > 0]  # filter valid

if len(tau) > 1:
    mean_delay = np.mean(tau)
    rms_ds = np.sqrt(np.mean((tau - mean_delay)**2))
    bc = 1.0 / (5.0 * rms_ds) if rms_ds > 0 else float('inf')
    
    print(f'Number of paths: {len(tau)}')
    print(f'Min delay: {tau.min()*1e9:.2f} ns')
    print(f'Max delay: {tau.max()*1e9:.2f} ns')
    print(f'Mean excess delay: {mean_delay*1e9:.2f} ns')
    print(f'RMS delay spread: {rms_ds*1e9:.2f} ns')
    print(f'Coherence bandwidth (0.5 corr): {bc/1e6:.2f} MHz')
    
    # Power delay profile
    plt.figure(figsize=(10, 4))
    plt.stem(tau * 1e9, np.ones_like(tau), linefmt='c-', markerfmt='co', basefmt='k-')
    plt.xlabel('Delay (ns)')
    plt.ylabel('Relative Power')
    plt.title('Power Delay Profile')
    plt.tight_layout()
    plt.show()
else:
    print('Not enough paths for analysis')

## Try Different Scenes

Replace `rt.scene.munich` with:
- `rt.scene.etoile` — Arc de Triomphe roundabout
- `rt.scene.simple_street_canyon` — idealized urban canyon
- `rt.scene.simple_wedge` — single diffracting edge

Experiment with different TX/RX positions, antenna configurations, and `max_depth` values.